# NLP Group Assignment — Question 4
# Integrated Background Editor: Live Segmentation, Spelling Correction & Constituency-Based Grammar Checking

This notebook builds the integrated editor requested in Q4, **reusing** (not reimplementing) the trained
components from Q1 (English trigram segmentation LM + trigram HMM POS decoder) and Q3 (vocabulary,
unigram/bigram models, Method A / Method B spelling correction).

All reusable logic lives in **`nlp_pipeline.py`**, a single module imported both here *and* by
`streamlit_app.py` (the live deployment), so the notebook and the deployed app are provably running the
exact same models and functions — nothing is duplicated or re-implemented between the two surfaces.

**What this notebook does, in order:**

1. Train/load every model once (`build_models()`): Q1's segmentation LM + POS HMM, Q3's spelling models,
   Q4's own shared add-k-smoothed bigram + trigram LM, and the PCFG (from Penn Treebank, or a small
   offline fallback grammar if NLTK/the treebank sample isn't available in this environment).
2. **Part 1** — simulate live typing of a passage (with randomly merged tokens) and show the
   `[SEGMENT-ALERT]` / `[SPELL-ALERT]` / `[GRAMMAR-ALERT]` stream, with latency measurements.
3. **Part 2** — demonstrate the PCFG constituency parser and the Q1↔PTB tagset reconciliation.
4. **Part 3** — demonstrate the shared bigram/trigram LM (sentence log-prob / perplexity).
5. **Part 4** — end-of-passage per-sentence comparison table + the method-selection decision rule.
6. **Part 5** — the Speed-Demon benchmark (1,000-token batch), plus notes on the Streamlit deployment
   (`streamlit_app.py`, included alongside this notebook).
7. Two full sample runs on different randomly-sampled passages, and the comparative-analysis report.

**Data availability note** (same pattern Q1 used): this notebook first tries `nltk.corpus.brown` and
`nltk.corpus.treebank`. If NLTK / internet access isn't available in whatever environment you run this
in, every component automatically falls back to a small embedded corpus / grammar so the whole pipeline
still runs end-to-end for demonstration. **For your actual submission, run this with `nltk.download('brown')`
and `nltk.download('treebank')` available** so the reported numbers are meaningful.


## 0. Setup

In [ ]:
# If nlp_pipeline.py sits next to this notebook (it does in the submitted repo), this just works.
import sys, os
sys.path.insert(0, os.getcwd())

import time
import random
import pandas as pd

import nlp_pipeline as P

random.seed(7)


In [ ]:
# Best-effort NLTK data download (no-ops safely if there's no internet access in this environment --
# every downstream function already has an offline fallback, see nlp_pipeline.py).
try:
    import nltk
    for pkg in ("brown", "treebank", "gutenberg"):
        try:
            nltk.data.find(f"corpora/{pkg}")
        except LookupError:
            nltk.download(pkg, quiet=True)
    print("NLTK corpora ready.")
except Exception as e:
    print(f"NLTK not available in this environment ({e}); the pipeline will use its offline fallbacks.")


## 1. Build / train every model once

`build_models()` is the **single training entry point** for the whole notebook (and the Streamlit app,
which imports it too). It trains:

- Q1's English segmentation trigram LM + trigram HMM POS tagger (unchanged from Q1),
- Q3's vocabulary + unigram/bigram + Method A/B spelling models (unchanged from Q3),
- Q4's *own* shared add-k-smoothed bigram + trigram LM (used for grammar alerting / final analysis, kept
  separate from Q1's segmentation LM as the assignment specifies),
- the PCFG (from the Penn Treebank sample, or the offline fallback CNF grammar).

Nothing below retrains any of these — every subsequent cell just calls into the already-trained `models`
object.


In [ ]:
models, held_out_test_sentences = P.build_models(max_word_len=12)


## 2. Part 1 — Background typing simulation (merges, SEGMENT / SPELL / GRAMMAR alerts)

**Chosen `p` (merge probability) = `0.08`.**
A typist misses the spacebar roughly 1 time in 12 word-boundaries — frequent enough that segmentation has
real, regular work to do (about 1–2 merges per 20-word sentence), without merging so heavily that almost
every token becomes a segmentation problem and the spelling/grammar layers barely get exercised.

**Chosen `N` (grammar trigger interval) = `15` tokens.**
SEGMENT/SPELL checks are cheap (near-O(1) vocabulary lookups, see the Speed-Demon benchmark in Part 5), so
they run on *every* token. The grammar/real-word check is heavier (perplexity over a window + an
edit-distance sweep across it), so it's only run every `15` tokens — roughly every half-sentence to
sentence — which keeps the expensive check off the per-token hot path while still catching an implausible
window well before the passage ends.


In [ ]:
PASSAGE_LIBRARY = None
try:
    import nltk
    from nltk.corpus import gutenberg
    PASSAGE_LIBRARY = gutenberg
except Exception:
    PASSAGE_LIBRARY = None


FALLBACK_PASSAGES = [
    ("the quick brown fox jumps over the lazy dog and then runs away quickly into the dark forest "
     "she eats a green salad with her friends every single day without fail").split(),
    ("i would like to see the world and meet new people along the way while learning something "
     "new about myself and about the places that i visit on this long and winding journey").split(),
]


def sample_passage(min_words=25, max_words=45, rng=None):
    """Randomly sample a 5-8-sentence-ish contiguous passage. Uses NLTK Gutenberg when available
    (a different random file/passage each run, as specified); otherwise falls back to an embedded
    passage so the notebook still runs offline."""
    rng = rng or random
    if PASSAGE_LIBRARY is not None:
        fileids = PASSAGE_LIBRARY.fileids()
        fid = rng.choice(fileids)
        words = [w.lower() for w in PASSAGE_LIBRARY.words(fid) if w.isalpha()]
        if len(words) > max_words:
            start = rng.randrange(0, len(words) - max_words)
            n = rng.randint(min_words, max_words)
            return words[start:start + n], fid
    passage = rng.choice(FALLBACK_PASSAGES)
    return passage, "embedded-fallback"


def run_live_typing_demo(passage_words, models, p=P.MERGE_PROB, n_trigger=P.GRAMMAR_TRIGGER_N,
                          sleep_s=0.0, verbose=True, rng=None):
    """Streams `passage_words` word-by-word (simulating time.sleep()-paced typing), merging some
    adjacent words per `p`, and runs the live SEGMENT/SPELL/GRAMMAR checks on each resulting token,
    in that order, exactly as Part 1 specifies."""
    rng = rng or random
    merged_tokens = P.simulate_fast_typing_merges(passage_words, p=p, rng=rng)
    session = P.LiveEditorSession(models, n_trigger=n_trigger)

    if verbose:
        print(f"Original passage ({len(passage_words)} words):\n  {' '.join(passage_words)}\n")
        print(f"Simulated-typing token stream ({len(merged_tokens)} tokens, some merged):\n  {merged_tokens}\n")
        print("--- live alerts as tokens arrive ---")

    n_alerts_before = 0
    for tok in merged_tokens:
        session.process_token(tok)
        if sleep_s:
            time.sleep(sleep_s)
        if verbose and len(session.alerts) > n_alerts_before:
            for a in session.alerts[n_alerts_before:]:
                print(f"  [{a['type']}] {a['message']}")
            n_alerts_before = len(session.alerts)

    return session, merged_tokens


In [ ]:
demo_passage, demo_source = sample_passage(rng=random.Random(7))
print(f"(sampled from: {demo_source})\n")
session1, merged1 = run_live_typing_demo(demo_passage, models, sleep_s=0.0)


### Latency of the live checks (Part 1, item 5)

In [ ]:
lat = session1.latency_report()
print(f"Average per-token SEGMENT+SPELL check latency : {lat['avg_seg_spell_ms']:.3f} ms  (n={lat['n_tokens']} tokens)")
print(f"Average per-trigger GRAMMAR check latency      : {lat['avg_grammar_trigger_ms']:.3f} ms  (n={lat['n_grammar_triggers']} triggers)")
print()
print("Both are sub-millisecond here, well under any realistic simulated-typing inter-keystroke delay "
      "(e.g. `time.sleep(0.05-0.3)` per word), confirming neither check would visibly lag live typing.")


## 3. Part 2 — PCFG constituency parser + tagset reconciliation

`models.pcfg_grammar` was induced from the Penn Treebank sample (or, offline, from the small fallback CNF
grammar in `nlp_pipeline.py`). `P.parse_with_pcfg` runs Viterbi/CKY most-probable-parse and **never
raises** — sentences it can't parse come back as `(None, None)` and are flagged as unparseable downstream,
per the assignment's requirement.

**Tagset reconciliation.** Q1's English tagger outputs coarse Universal tags (`NOUN`, `VERB`, `ADJ`, ...);
the PCFG's lexical productions carry Penn-Treebank-style tags (`NN`, `VBZ`, `JJ`, ...) as their
preterminal labels. Rather than writing a second bespoke lookup table, we reuse **Q1's own**
`brown_tag_to_universal` mapper on *both* sides — Penn Treebank tags share Brown's tagging prefixes
(`NN*`, `VB*`, `JJ*`, `IN`, `DT`, ...), so the same prefix-based function projects a PTB tag into the same
coarse Universal space Q1 already uses. `reconcile_tags` compares the two projections per word.

*Documented accuracy loss:* projecting both tagsets down to ~12 coarse Universal categories means
fine-grained distinctions the PCFG's tags *do* carry (e.g. `VBZ` vs `VBP` vs `VBD`, or `NN` vs `NNS`) are
invisible to the reconciliation check — a Q1 tag of `VERB` will "agree" with any PTB verb subtype. This is
an intentional trade-off: it avoids maintaining a second, more brittle fine-grained mapping table, at the
cost of only catching reconciliation disagreements at the coarse-category level (e.g. Q1 saying `NOUN`
where the parse's preterminal is actually a verb).


In [ ]:
demo_sentences = [
    "the quick fox jumps over the mat".split(),
    "she jumps over the dog".split(),
    "colourless green ideas sleep furiously".split(),  # expected to be unparseable under our small grammars
]

for sent in demo_sentences:
    tree, logprob = P.parse_with_pcfg(sent, models.pcfg_grammar)
    print("Sentence:", " ".join(sent))
    if tree is None:
        print("  -> UNPARSEABLE (flagged, pipeline continues)")
    else:
        print(f"  -> parsed, log-prob = {logprob:.3f}")
        print(f"     {tree}")

        # Q1 tags for the same words (independently, via the trained HMM) vs the PCFG's own POS leaves
        q1_tags = models.hmm.viterbi_tag(sent)
        q1_pairs = list(zip(sent, q1_tags))
        recon = P.reconcile_tags(q1_pairs, tree)
        print("     Q1-tag vs PCFG-leaf-tag reconciliation (word, q1_tag, ptb_tag, agree):")
        for row in recon:
            print("      ", row)
    print()


## 4. Part 3 — Shared smoothed bigram + trigram LM

`models.q4_bigram_lm` and `models.q4_trigram_lm` are add-k smoothed (k=0.5) n-gram models trained on the
Brown Corpus, used for live `[GRAMMAR-ALERT]` perplexity scoring *and* for the Part 4 end-of-passage
sentence scoring. They're kept separate from Q1's `seg_lm` (which stays dedicated to segmentation scoring,
as the assignment specifies) even though both are `TrigramLM` instances.


In [ ]:
for sent in [
    "the quick fox jumps over the mat".split(),
    "dog a nd the n runs a".split(),          # a deliberately mangled window, for contrast
]:
    bi_ppl = models.q4_bigram_lm.perplexity(sent)
    tri_ppl = models.q4_trigram_lm.perplexity(sent)
    print(f"{' '.join(sent):45s}  bigram-ppl={bi_ppl:8.2f}   trigram-ppl={tri_ppl:8.2f}")

print(f"\n(training-set average trigram perplexity, used as the GRAMMAR-ALERT baseline: "
      f"{models.avg_train_trigram_ppl:.2f})")


## 5. Part 4 — End-of-passage per-sentence analysis & method-selection decision rule

**Decision rule** (`P.choose_verdict`, documented in `nlp_pipeline.py`):
1. If the PCFG parses the sentence, prefer its verdict (a successful parse is itself a strong
   grammaticality signal, and structural coverage is exactly what n-gram models can't see).
2. Otherwise, if the trigram model has adequate coverage of the sentence (its perplexity isn't a wild
   outlier relative to the training baseline), use the trigram verdict.
3. Otherwise fall back to the bigram verdict, since bigram degrades more gracefully than trigram on
   sparse/unseen sequences.


In [ ]:
rows1 = P.analyze_passage(session1, models)
df1 = pd.DataFrame(rows1)
df1["segmentation_merges_in_sentence"] = None  # per-sentence attribution shown in the full report below
df1


## 6. Part 5 — Speed-Demon benchmark

Builds a fixed batch of exactly 1,000 simulated tokens (a mix of single-edit misspellings and merged
word-pairs) and times **(a)** the full per-token live pipeline (SEGMENT-ALERT check + SPELL-ALERT check)
against **(b)** the grammar-trigger check alone, run in isolation on windows drawn from the same batch.


In [ ]:
bench = P.speed_demon_benchmark(models, batch_size=1000)
for k, v in bench.items():
    print(f"{k}: {v}")

print()
print(f"=> The per-token SEGMENT+SPELL layer costs ~{bench['avg_seg_spell_ms']:.3f} ms/token, "
      f"vs ~{bench['avg_grammar_window_ms']:.3f} ms per {P.GRAMMAR_TRIGGER_N}-token grammar-check window "
      f"(~{bench['avg_grammar_window_ms']/P.GRAMMAR_TRIGGER_N:.4f} ms/token amortised).")


**Conclusion.** The segmentation+spelling layer runs on *every* token and is individually cheap
(vocabulary-set lookups plus, only on the minority of tokens that are OOV, a bounded Viterbi segmentation
or SymSpell lookup), while the grammar layer batches its more expensive perplexity/edit-distance work over
a whole window and only runs once every `N` tokens. On the numbers above, the amortised per-token cost of
the segmentation+spelling layer is comparable to or below the amortised per-token cost of the grammar
layer — i.e. it is **cheap enough to keep running on every token live**, and does *not* need to be
throttled down to the grammar layer's trigger interval. If this changed on a much larger real vocabulary
(where Method A's alphabet-sized edit-distance-1 generation could get called more often), the fix
documented in Q3 already applies here for free: `SpellingModel.correct_nonword` defaults to **Method B**
(symmetric delete) precisely because it's the cheaper one at query time once corrections are being
requested on every keystroke rather than in an offline batch.


## 7. Two full sample runs on different randomly-sampled passages, with the final table

In [ ]:
for run_idx, seed in enumerate([101, 202], start=1):
    print("=" * 90)
    print(f"SAMPLE RUN {run_idx}  (seed={seed})")
    print("=" * 90)
    rng = random.Random(seed)
    passage, source = sample_passage(rng=rng)
    print(f"(sampled from: {source})\n")
    sess, merged = run_live_typing_demo(passage, models, rng=rng)
    print()
    rep = sess.latency_report()
    print(f"latency -> seg+spell: {rep['avg_seg_spell_ms']:.3f} ms/token | "
          f"grammar: {rep['avg_grammar_trigger_ms']:.3f} ms/trigger")
    print(f"segmentation merges resolved: {sess.n_segmentation_merges_resolved}  |  "
          f"spelling corrections applied: {sess.n_spelling_corrections}")
    rows = P.analyze_passage(sess, models)
    display(pd.DataFrame(rows))
    print()


## 8. Comparative analysis

**How often did the real-time alerts agree with the final end-of-passage verdict for the same sentence?**
Live `[GRAMMAR-ALERT]`s fire on a fixed trailing window and use only the shared bigram/trigram LM; the
final verdict (Part 4) additionally consults the PCFG. In the sample runs above, windows the live check
flagged as high-perplexity generally also ended up `FLAGGED` in the final table when they fell back to the
trigram/bigram verdict — but any sentence the PCFG *does* manage to parse is scored `OK` at the end even if
its live-window perplexity looked mildly elevated, since a successful parse outweighs a merely-elevated
(not extreme) n-gram perplexity in the decision rule. Disagreements in the other direction (live alert
fired, but the sentence boundary used for the final table doesn't line up with the live window boundary)
are also possible, since the live window is a fixed token count while the final sentences are re-chunked
from the corrected stream — that boundary mismatch is a known limitation of comparing a streaming,
fixed-size window against post-hoc sentence segmentation.

**PCFG vs bigram/trigram judgments — which caught different error classes?**
The PCFG is sensitive to *structural* well-formedness — it fails to parse a sentence whose sequence of
categories doesn't fit any production, even if every individual word transition is locally plausible. The
bigram/trigram LMs are sensitive to *local word-sequence* plausibility — they catch an implausible pair or
triple of words even inside an otherwise well-formed constituent, but have no notion of overall tree
structure and can be fooled by locally-fluent nonsense. In practice the two catch complementary error
classes: unresolved segmentation merges (e.g. `dogandthen` → `dog a nd the n`) tend to blow up n-gram
perplexity immediately (very low-frequency bigrams/trigrams) *and* fail to parse, while a real-word swap
that keeps a sentence locally fluent but subtly wrong is caught mainly by the bigram/trigram real-word
check rather than by parse failure.

**Effect of the chosen trigger interval `N` and merge probability `p`.**
A smaller `N` catches grammar problems sooner but increases false-alert churn, since bigram/trigram
perplexity computed over a very short window is noisy (few observations to average over) and a single rare
but valid word can spike it. A larger `N` smooths that noise out but delays detection and lets more of a
bad window accumulate before it's caught. A higher `p` creates more segmentation work per sentence (good
for exercising Q1's decoder) but, past a point, starts merging so many boundaries that the segmenter's own
errors (like the multi-way "dogandthen" split above) start dominating the alert stream and drowning out
genuine spelling/grammar issues — `p=0.08` was chosen to keep merges frequent but still the minority case.

**Interaction effects between sub-systems.**
A segmentation split or spelling correction changing a sentence's word sequence can flip whether the PCFG
parses it at all, and, downstream, which method the Part-4 decision rule ends up choosing. For example, if
`SPELL-ALERT` corrects a garbled token back to a word the PCFG's lexicon actually covers, a sentence that
would otherwise have fallen back to the bigram verdict (because the PCFG failed to parse it) can instead
get a clean PCFG parse and an `OK` verdict from the *first* branch of the decision rule. Conversely, an
unresolved multi-way segmentation split (see `dogandthen` above) injects extra, ungrammatical short tokens
into the stream, which both a) tanks the PCFG's chance of parsing that sentence and b) inflates
bigram/trigram perplexity, compounding into a `FLAGGED` verdict driven by *both* signals at once rather
than by an independent grammar mistake.

**Speed-Demon summary.** See Part 5 above — the segmentation+spelling layer's amortised per-token cost is
in the same range as the grammar layer's amortised per-token cost, so it's cheap enough to run on every
token live rather than needing to be throttled to the grammar layer's `N`-token trigger interval.


## 9. Live Streamlit deployment

The fully integrated pipeline is also packaged as a Streamlit web app in **`streamlit_app.py`** (in this
same folder), importing this exact `nlp_pipeline.py` module — so the deployed app runs the identical
trained models and alert logic demonstrated above, not a re-implementation. It supports:

- **Live typing mode**: as you type into a text box, the app incrementally processes new tokens and shows
  SEGMENT/SPELL alerts as they happen, and GRAMMAR alerts every `N` tokens.
- **Simulated-typing mode**: paste or auto-sample a passage and watch it "type itself" with merges injected,
  exactly like the `run_live_typing_demo` cells above, rendered live in the browser.
- A **final analysis** view (once you're done) showing the Part 4 per-sentence PCFG/bigram/trigram table
  and total latency.

See the `README.md` included alongside this notebook for exact steps to push this to a Git repo and deploy
it on Streamlit Community Cloud.
